# Milvus retrieval (use existing `flat.db` and `ivf_flat.db`)

You already encoded + stored the corpus vectors in Milvus. This notebook only:
- encodes **queries** (required to search)
- runs **retrieval only** against the existing DB files
- reports **retrieval time** + metrics


In [1]:
import json
import logging
import time
import pandas as pd
from pymilvus import MilvusClient

from financerag.retrieval import SentenceTransformerEncoder
from financerag.tasks.BaseTask import BaseTask

logging.basicConfig(level=logging.INFO)


/opt/homebrew/anaconda3/envs/FinRAG/lib/python3.11/site-packages/pymilvus/client/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/opt/homebrew/anaconda3/envs/FinRAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
QUERY_PATH = "/Users/vikashpr/Dev/Python/FinanceRAG/icaif-24-finance-rag-challenge/finder_queries.jsonl/queries.jsonl"
QRELS_PATH = "/Users/vikashpr/Dev/Python/FinanceRAG/icaif-24-finance-rag-challenge/FinDER_qrels.tsv"

# Existing Milvus Lite DB files (already contain encoded corpus vectors)
MILVUS_FLAT_DB = "./milvus_db/flat.db"
MILVUS_IVF_FLAT_DB = "./milvus_db/ivf_flat.db"

TOP_K = 100


In [3]:
def load_jsonl(path):
    with open(path, "r") as f:
        for line in f:
            yield json.loads(line)

queries = {q["_id"]: q["text"] for q in load_jsonl(QUERY_PATH)}
df = pd.read_csv(QRELS_PATH, sep="\t")
qrels = df.groupby("query_id").apply(lambda g: dict(zip(g["corpus_id"], g["score"])), include_groups=False).to_dict()

print(f"Queries: {len(queries)}")
print(f"Qrels queries: {len(qrels)}")


Queries: 216
Qrels queries: 64


In [4]:
# IMPORTANT: corpus is already in Milvus; encoder is only for query vectors
encoder = SentenceTransformerEncoder(
    model_name_or_path="intfloat/e5-large-v2",
    query_prompt="query: ",
    doc_prompt="passage: ",
)

query_ids = list(queries.keys())
query_texts = [queries[qid] for qid in query_ids]

t0 = time.time()
query_embeddings = encoder.encode_queries(query_texts, batch_size=64)
query_encoding_time_s = time.time() - t0

print(f"Query encoding time: {query_encoding_time_s:.2f}s")


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: intfloat/e5-large-v2
Batches: 100%|██████████| 4/4 [00:01<00:00,  2.03it/s]

Query encoding time: 1.98s


In [5]:
def pick_collection(client: MilvusClient) -> str:
    cols = client.list_collections() or []
    if not cols:
        raise ValueError("No collections found in this Milvus DB.")
    if len(cols) == 1:
        return cols[0]
    # If multiple, choose the one with most rows
    best = None
    best_rows = -1
    for c in cols:
        try:
            rows = int(client.get_collection_stats(collection_name=c).get("row_count", 0))
        except Exception:
            rows = 0
        if rows > best_rows:
            best, best_rows = c, rows
    return best

def retrieval_only(db_uri: str, top_k: int, search_params: dict | None = None):
    client = MilvusClient(uri=db_uri)
    try:
        collection = pick_collection(client)
        rows = int(client.get_collection_stats(collection_name=collection).get("row_count", 0))
        print(f"DB={db_uri} | collection={collection} | rows={rows}")
        
        params = {"metric_type": "COSINE", "params": search_params or {}}
        results = {}
        t0 = time.time()
        for i, qid in enumerate(query_ids):
            qvec = query_embeddings[i].tolist()
            hits = client.search(
                collection_name=collection,
                data=[qvec],
                limit=top_k,
                output_fields=["doc_id"],
                search_params=params,
            )
            scores = {}
            for hit in hits[0]:
                doc_id = hit.get("entity", {}).get("doc_id") or hit.get("id")
                scores[str(doc_id)] = float(hit.get("distance", 0.0))
            results[qid] = scores
        retrieval_time_s = time.time() - t0
        return results, {
            'collection': collection,
            'rows': rows,
            'retrieval_time_s': retrieval_time_s,
            'avg_query_ms': (retrieval_time_s/len(query_ids))*1000,
        }
    finally:
        try:
            client.close()
        except Exception:
            pass


## FLAT (`flat.db`) retrieval


In [6]:
flat_results, flat_timing = retrieval_only(MILVUS_FLAT_DB, top_k=TOP_K, search_params={})
print(f"Retrieval time: {flat_timing['retrieval_time_s']:.2f}s | avg/query: {flat_timing['avg_query_ms']:.2f}ms")

flat_ndcg, flat_map, flat_recall, flat_precision = BaseTask.evaluate(
    qrels=qrels, results=flat_results, k_values=[1, 5, 10]
)
print('NDCG', flat_ndcg)
print('MAP', flat_map)
print('Recall', flat_recall)
print('Precision', flat_precision)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DB=./milvus_db/flat.db | collection=corpus_flat | rows=13863


INFO:financerag.tasks.BaseTask:For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:NDCG@1: 0.3125
INFO:financerag.tasks.BaseTask:NDCG@5: 0.3860
INFO:financerag.tasks.BaseTask:NDCG@10: 0.4281
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:MAP@1: 0.2531
INFO:financerag.tasks.BaseTask:MAP@5: 0.3465
INFO:financerag.tasks.BaseTask:MAP@10: 0.3685
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:Recall@1: 0.2531
INFO:financerag.tasks.BaseTask:Recall@5: 0.4594
INFO:financerag.tasks.BaseTask:Recall@10: 0.5742
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:P@1: 0.3125
INFO:financerag.tasks.BaseTask:P@5: 0.1219
INFO:financerag.tasks.BaseTask:P@10: 0.0797


Retrieval time: 0.88s | avg/query: 4.07ms
NDCG {'NDCG@1': 0.3125, 'NDCG@5': 0.38599, 'NDCG@10': 0.42814}
MAP {'MAP@1': 0.25312, 'MAP@5': 0.34648, 'MAP@10': 0.36851}
Recall {'Recall@1': 0.25312, 'Recall@5': 0.45937, 'Recall@10': 0.57422}
Precision {'P@1': 0.3125, 'P@5': 0.12188, 'P@10': 0.07969}


## IVF_FLAT (`ivf_flat.db`) retrieval


In [7]:
ivf_results, ivf_timing = retrieval_only(MILVUS_IVF_FLAT_DB, top_k=TOP_K, search_params={'nprobe': 10})
print(f"Retrieval time: {ivf_timing['retrieval_time_s']:.2f}s | avg/query: {ivf_timing['avg_query_ms']:.2f}ms")

ivf_ndcg, ivf_map, ivf_recall, ivf_precision = BaseTask.evaluate(
    qrels=qrels, results=ivf_results, k_values=[1, 5, 10]
)
print('NDCG', ivf_ndcg)
print('MAP', ivf_map)
print('Recall', ivf_recall)
print('Precision', ivf_precision)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DB=./milvus_db/ivf_flat.db | collection=corpus_ivf_flat | rows=13863


INFO:financerag.tasks.BaseTask:For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:NDCG@1: 0.3125
INFO:financerag.tasks.BaseTask:NDCG@5: 0.3860
INFO:financerag.tasks.BaseTask:NDCG@10: 0.4281
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:MAP@1: 0.2531
INFO:financerag.tasks.BaseTask:MAP@5: 0.3465
INFO:financerag.tasks.BaseTask:MAP@10: 0.3685
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:Recall@1: 0.2531
INFO:financerag.tasks.BaseTask:Recall@5: 0.4594
INFO:financerag.tasks.BaseTask:Recall@10: 0.5742
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:P@1: 0.3125
INFO:financerag.tasks.BaseTask:P@5: 0.1219
INFO:financerag.tasks.BaseTask:P@10: 0.0797


Retrieval time: 0.88s | avg/query: 4.06ms
NDCG {'NDCG@1': 0.3125, 'NDCG@5': 0.38599, 'NDCG@10': 0.42814}
MAP {'MAP@1': 0.25312, 'MAP@5': 0.34648, 'MAP@10': 0.36851}
Recall {'Recall@1': 0.25312, 'Recall@5': 0.45937, 'Recall@10': 0.57422}
Precision {'P@1': 0.3125, 'P@5': 0.12188, 'P@10': 0.07969}


In [8]:
summary = pd.DataFrame([
    {
        'db': 'flat.db',
        'collection': flat_timing['collection'],
        'rows': flat_timing['rows'],
        'retrieval_s': round(flat_timing['retrieval_time_s'], 3),
        'avg_query_ms': round(flat_timing['avg_query_ms'], 3),
        'NDCG@10': flat_ndcg['NDCG@10'],
    },
    {
        'db': 'ivf_flat.db',
        'collection': ivf_timing['collection'],
        'rows': ivf_timing['rows'],
        'retrieval_s': round(ivf_timing['retrieval_time_s'], 3),
        'avg_query_ms': round(ivf_timing['avg_query_ms'], 3),
        'NDCG@10': ivf_ndcg['NDCG@10'],
    },
])

print(f"Query encoding time (once): {query_encoding_time_s:.2f}s")
print(summary.to_string(index=False))


Query encoding time (once): 1.98s
         db      collection  rows  retrieval_s  avg_query_ms  NDCG@10
    flat.db     corpus_flat 13863        0.879         4.068  0.42814
ivf_flat.db corpus_ivf_flat 13863        0.878         4.064  0.42814
